# AI Engineer Assistant Chatbot (Gemma 3, Multi-Turn)

This notebook builds a multi-turn conversational assistant on top of a local Gemma 3 model,
specialized (via system prompt) as a senior AI engineer assistant: it diagnoses problems,
explains the reasoning behind its recommendations, flags likely pitfalls, and gives concrete
runnable code rather than only abstract advice.

Unlike a single-shot Q&A call, this notebook maintains a running `chat_history` across calls to
`ask()`, so follow-up questions have the context of everything said before in the conversation
(and can optionally include an image in any turn).

**Requirements:**
- A Hugging Face token with access to the gated `google/gemma-3-4b-it` model
- A GPU runtime is strongly recommended for running the 4B-parameter model

⚠️ **Security note:** Never commit a real API token into this notebook. Use an environment
variable or Colab secret (`login(token=os.environ["HF_TOKEN"])`) instead of hardcoding it.


## 1. Install Dependencies

In [ ]:
!pip install -q transformers accelerate bitsandbytes huggingface_hub datasets peft trl


## 2. Hugging Face Authentication

Model page: https://huggingface.co/google/gemma-3-4b-it

This model is **gated** — request access on the model page and authenticate with a Hugging Face
token before it can be downloaded.


In [ ]:
from huggingface_hub import login

# Set your Hugging Face token as an environment variable / Colab secret instead of hardcoding
# it here. Example: login(token=os.environ["HF_TOKEN"])
login(token="")


## 3. Load the Model & Run a Sanity-Check Query

Loads the Gemma 3 processor and model, then runs a single multimodal test query (image + text)
to confirm everything loaded correctly before building the multi-turn chatbot on top of it.


In [ ]:
from transformers import AutoProcessor, AutoModelForMultimodalLM

MODEL_ID = "google/gemma-3-4b-it"

processor = AutoProcessor.from_pretrained(MODEL_ID)
model = AutoModelForMultimodalLM.from_pretrained(MODEL_ID, device_map="auto")

# Quick sanity check: ask the model a simple multimodal question using a sample image
messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "url": "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/p-blog/candy.JPG"},
            {"type": "text", "text": "What animal is on the candy?"}
        ]
    },
]
inputs = processor.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
).to(model.device)

outputs = model.generate(**inputs, max_new_tokens=40)
print(processor.decode(outputs[0][inputs["input_ids"].shape[-1]:]))


## 4. Multi-Turn Assistant

Defines the system prompt that specializes the model as a senior AI engineer assistant, and an
`ask()` helper that:
- Appends the new user message (and optional image) to a persistent `chat_history`
- Re-runs the full conversation through the model so it has all prior context
- Appends the model's reply back into `chat_history` for the next turn

Because `chat_history` is a module-level list, each call to `ask()` continues the same
conversation rather than starting fresh.


In [ ]:
SYSTEM_PROMPT = (
    "You are a senior AI engineer assistant. You have deep expertise in machine learning, "
    "model architectures, fine-tuning, deployment, and debugging AI systems. "
    "When helping users, you: "
    "- Diagnose problems precisely before suggesting fixes "
    "- Explain the 'why' behind technical recommendations, not just the 'what' "
    "- Point out potential pitfalls (memory issues, version mismatches, edge cases) proactively "
    "- Give concrete, runnable code when relevant, not just abstract advice "
    "- Ask clarifying questions when the problem is ambiguous, rather than guessing "
    "Keep answers technically precise but clear."
)

chat_history = [{"role": "system", "content": [{"type": "text", "text": SYSTEM_PROMPT}]}]


def ask(user_message, image_url=None, max_new_tokens=512):
    """
    Sends a new message to the assistant, optionally including an image, and returns
    its reply. Both the user's message and the assistant's reply are appended to the
    persistent `chat_history`, so subsequent calls continue the same conversation
    with full prior context.
    """
    content = []
    if image_url:
        content.append({"type": "image", "url": image_url})
    content.append({"type": "text", "text": user_message})

    chat_history.append({"role": "user", "content": content})

    inputs = processor.apply_chat_template(
        chat_history, add_generation_prompt=True, tokenize=True,
        return_dict=True, return_tensors="pt",
    ).to(model.device)

    output = model.generate(**inputs, max_new_tokens=max_new_tokens)
    reply = processor.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

    chat_history.append({"role": "assistant", "content": [{"type": "text", "text": reply}]})
    return reply


## 5. Example Query

In [ ]:
print(ask("What's the difference between LoRA and full fine-tuning?"))
